# W10 — Matplotlib + Seaborn: od wykresu do dashboardu

**Programowanie w Pythonie II** | Wykład 10
**Politechnika Opolska** | Analityka danych w biznesie | Semestr 2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sp6jaz/python2-materialy/blob/master/dzienne/tydzien-10/wyklad/seaborn_dashboard_demo.ipynb)

> **Dwie ścieżki pracy z notebookiem:**
> - **Lokalnie** (zalecane do długotrwałej pracy): pobierz plik, otwórz w VS Code, uruchom w aktywowanym środowisku `.venv`. Pełna kontrola nad bibliotekami, dostęp do plików, brak limitów.
> - **W przeglądarce** (Google Colab — kliknij badge powyżej): otwiera się w ciągu 5 sekund, środowisko gotowe (seaborn, matplotlib, pandas już zainstalowane). Działa nawet z telefonu. Idealne do szybkiego podglądu i eksperymentowania.

**Temat dzisiejszego wykładu:** Seaborn (wygodne wykresy statystyczne) + matplotlib `GridSpec` (nieregularne układy) = profesjonalny dashboard analityczny.

**Dataset przewodni:** `tips` (244 rachunki z amerykańskiej restauracji, wbudowany w seaborn — `sns.load_dataset('tips')`).

## 🎯 Pięć kluczowych zagadnień tego wykładu

Pięć fundamentów, na których stoi cały materiał. Wszystko inne w notebooku to ich rozwinięcie i przykłady — gdy się zgubisz, wróć tutaj.

🔑 **1. Kiedy seaborn, kiedy matplotlib** — seaborn = wygodny *wrapper* (nakładka) na matplotlib dla wykresów statystycznych; matplotlib = silnik pod spodem + pełna kontrola. **Seaborn nie zastępuje matplotlib — siedzi NA matplotlib.**

🔑 **2. Wybór typu wykresu pod pytanie** — porównanie wartości → *barplot* (wykres słupkowy); rozkład → *boxplot* (skrzynka z wąsami) / *violinplot* (wykres skrzypcowy); korelacja → *heatmap* (mapa cieplna); zależność dwóch zmiennych → *scatterplot* (wykres rozrzutu). **Nie dobierasz „najładniejszy" — dobierasz odpowiadający na pytanie.**

🔑 **3. Long (długi) vs Wide (szeroki) format danych** — seaborn lubi long (`data=df, x=, y=, hue=`); matplotlib lubi wide (`ax.bar(x, y)`). **Konwersja `pivot()`/`melt()` to przygotowanie danych przed wizualizacją.**

🔑 **4. Dashboard (kokpit menedżerski — jeden raport z wieloma wykresami) = jedna figura, wiele wykresów, jedna historia** — `GridSpec` daje nieregularne panele, `subplots()` daje regularną siatkę. **Dashboard nie jest kolażem — jest narracją.**

🔑 **5. Eksport profesjonalny** — `savefig(..., dpi=150, bbox_inches='tight', facecolor='white')`. **Bez tych 3 parametrów wykres nie wygląda profesjonalnie.**

> Wszystko inne w tym notebooku jest implementacją tych 5 idei. Gdy się zgubisz — wróć tutaj.

## Efekty uczenia się

Po tym wykładzie osoba studiująca:

1. **Wybiera** odpowiedni typ wykresu seaborn (`barplot`/`boxplot`/`violinplot`/`heatmap`/`scatterplot`/`pairplot` — wykres macierzy par, każda zmienna vs każda) do pytania analitycznego, uzasadniając wybór charakterem danych
2. **Tłumaczy** różnicę między matplotlib a seaborn na poziomie architektury (*wrapper* — nakładka programowa, *grammar of graphics* — gramatyka grafiki, automatyczne statystyki)
3. **Konstruuje** wielopanelowy dashboard używając `GridSpec` z nieregularnymi panelami i wspólnymi osiami
4. **Projektuje** spójną narrację dashboardu (kolejność paneli, jednolita paleta, główne pytanie biznesowe)
5. **Eksportuje** wizualizację z parametrami publikacyjnymi (`dpi`, `bbox_inches`, `facecolor`, format)
6. **Identyfikuje** typowe pułapki (`pairplot` nie przyjmuje `ax=`, `hue` z `split=True` wymaga 2 kategorii, `observed=True` w `groupby`) i unika ich

**Czym się różnimy od W09:** W09 nauczyło Cię rysować pojedynczy wykres w matplotlib. Dziś składamy z tych pojedynczych wykresów **dashboard** — dokument, który menedżer czyta w 30 sekund i zna stan biznesu.

## 0. Setup (konfiguracja środowiska) — biblioteki i *dataset* (zbiór danych)

> **Powtórka z W09:** `%matplotlib inline` to *magic command* (komenda specjalna Jupytera) renderująca wykresy w notebooku. `import matplotlib.pyplot as plt` to *alias* (skrót, zwyczajowa nazwa) konwencjonalny w społeczności data science. Aliasy: `sns` (seaborn), `plt` (matplotlib.pyplot), `pd` (pandas), `np` (numpy).

In [ ]:
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np

# Globalny motyw — wszystkie wykresy w tym stylu (raz, na początku notebooka)
sns.set_theme(style='whitegrid', palette='muted')

# Dataset przewodni — wbudowany w seaborn (244 rachunki z restauracji)
tips = sns.load_dataset('tips')

print(f"Wersje:")
print(f"  seaborn   : {sns.__version__}")
print(f"  matplotlib: {plt.matplotlib.__version__}")
print(f"  pandas    : {pd.__version__}")
print(f"\nDataset tips: {tips.shape[0]} wierszy x {tips.shape[1]} kolumn")
print(f"\nKolumny:")
for col in tips.columns:
    print(f"  {col:12s} : {tips[col].dtype}")
print(f"\nPierwsze 3 wiersze:")
print(tips.head(3).to_string())

**Co właśnie zrobiliśmy:**

- `sns.set_theme(style='whitegrid', palette='muted')` — ustawia globalny *motyw* (theme — wygląd) seaborn. Style (`style=`): `whitegrid` (dyskretna siatka — czytelne do prezentacji), `darkgrid` (ciemne tło), `white` (minimalistyczne, do publikacji), `ticks` (techniczne).
- `tips = sns.load_dataset('tips')` — wbudowany *dataset* (zbiór danych) — *cache* (pamięć podręczna) lokalny po pierwszym pobraniu. 7 kolumn: rachunek, napiwek, płeć płacącego, sekcja palaczy, dzień, pora dnia, liczba osób przy stoliku.
- `plt.matplotlib.__version__` — sprawdzamy wersję (pułapka: NIE `plt.__version__`, bo `pyplot` nie ma własnego *atrybutu* — pola w obiekcie — wersji).

> **Ciekawostka.** Nazwa **Seaborn** pochodzi od postaci Sama Seaborna z amerykańskiego serialu „The West Wing" (Prezydencki poker). Twórca biblioteki Michael Waskom (PhD z Stanforda, neurokognitywista) po prostu lubił ten serial. Najbardziej wpływowa biblioteka wizualizacji w data science nazwana od fikcyjnego doradcy prezydenckiego.

> **Pułapka praktyczna:** jeśli `sns.load_dataset('tips')` rzuci błąd `URLError` lub timeout — to znak, że nie masz internetu. Dataset jest pobierany z GitHub przy pierwszym uruchomieniu i potem działa lokalnie. Workaround (obejście): w laboratorium uczelnianym może być potrzebne ponowne uruchomienie z aktywnym połączeniem internetowym.

## 0.1 Dlaczego seaborn? — Słupek bez kontekstu kłamie

Zanim zaczniemy, jeden eksperyment. **W09 poznałeś `plt.bar()`** — pokazuje wartość. Ale czy *sama wartość* wystarczy do podjęcia decyzji biznesowej?

Wyobraź sobie raport: „średni napiwek w poniedziałek wynosi 3.50, a we wtorek 3.20 — różnica 9%". Menedżer pyta: czy to różnica istotna, czy *szum statystyczny* (przypadkowa fluktuacja przy małej próbce, nie prawdziwa różnica)?

Sam słupek **nie odpowiada** na to pytanie. Potrzebujemy *miary niepewności* (jak bardzo można ufać oszacowaniu) — *przedziału ufności* (CI — *confidence interval*; zakres w którym z 95% prawdopodobieństwem leży prawdziwa średnia). Matplotlib rysuje co podasz; seaborn liczy CI automatycznie.

In [ ]:
# Eksperyment: ten sam dataset, dwa wykresy. Który jest uczciwy?
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# LEWY: matplotlib — sama średnia, bez kontekstu
day_means = tips.groupby('day', observed=True)['total_bill'].mean()
axes[0].bar(day_means.index.astype(str), day_means.values, color='#4c72b0')
axes[0].set_title('matplotlib: tylko średnia\n(bez informacji o niepewności)', fontsize=11)
axes[0].set_xlabel('Dzień tygodnia')
axes[0].set_ylabel('Średni rachunek (USD)')

# PRAWY: seaborn — średnia + 95% CI (przedział ufności)
sns.barplot(data=tips, x='day', y='total_bill', ax=axes[1], color='#4c72b0',
            errorbar=('ci', 95))
axes[1].set_title('seaborn: średnia + 95% CI\n(wąsy = niepewność oszacowania)', fontsize=11)
axes[1].set_xlabel('Dzień tygodnia')
axes[1].set_ylabel('Średni rachunek (USD)')

fig.suptitle('Słupek bez kontekstu kłamie — uczciwość statystyczna seaborn',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
plt.close()

# Porównanie: ile obserwacji per dzień?
print(tips.groupby('day', observed=True)['total_bill'].agg(['count', 'mean', 'std']).round(2))

**Co tu właśnie zobaczyłeś:**

Lewy wykres pokazuje 4 słupki bez kontekstu — wyglądają autorytatywnie, ale **nie mówią o niepewności**. Prawy wykres ma wąsy (*error bars* — paski błędu): widać, że dla piątku wąs jest *szeroki* (mała próbka — n=19) i piątek może mieć większy lub mniejszy średni rachunek niż czwartek; dla soboty wąs jest *wąski* (n=87 — duża próbka), więc oszacowanie jest pewniejsze.

Ten sam wykres słupkowy + jeden parametr seaborn = **dramatyczna różnica w jakości decyzji biznesowej**.

> **To jest motywacja do całego dziś wykładu.** Seaborn nie robi „ładniejszych" wykresów — robi **uczciwiejsze** wykresy. Każdy zaawansowany typ wykresu (boxplot, violinplot, heatmap) odpowiada na pytanie, którego nie umie odpowiedzieć surowy matplotlib.

## 1. Model obiektowy matplotlib — Figure ↔ Axes

W W09 widziałeś dwa style: `plt.bar(...)` (krótki) i `fig, ax = plt.subplots(); ax.bar(...)` (długi). Oba działają — ale to nie są dwa „style". To **dwa różne API tego samego modelu obiektowego**, a model jest jeden:

```
┌────────────────────────────────────────────┐
│ Figure  (cały rysunek, jeden plik PNG)     │
│  ┌────────────┐    ┌────────────┐          │
│  │  Axes #1   │    │  Axes #2   │          │
│  │ (panel 1)  │    │ (panel 2)  │          │
│  │  bar plot  │    │  scatter   │          │
│  └────────────┘    └────────────┘          │
│  ┌────────────┐    ┌────────────┐          │
│  │  Axes #3   │    │  Axes #4   │          │
│  │  heatmap   │    │  box plot  │          │
│  └────────────┘    └────────────┘          │
│       fig.suptitle('Tytuł całości')        │
└────────────────────────────────────────────┘
```

**`Figure`** to ramka obrazu w galerii. **`Axes`** to poszczególne obrazy w ramce. Można mieć jeden obraz na całą ramkę, albo 4 obrazy (subplots), albo 6 obrazów różnej wielkości (`GridSpec`).

**Tytuł całości** to `fig.suptitle()` (super-title — nadtytuł), tytuł pojedynczego panelu to `ax.set_title()`.

In [ ]:
# Sprawdzamy hierarchię obiektów
fig, axes = plt.subplots(2, 2, figsize=(8, 4))

print(f"type(fig)        : {type(fig).__name__}")
print(f"type(axes)       : {type(axes).__name__}  (kontener)")
print(f"type(axes[0, 0]) : {type(axes[0, 0]).__name__}  (pojedynczy panel)")
print(f"\nfig zawiera {len(fig.axes)} Axes (po jednym na panel)")
print(f"axes.shape       : {axes.shape}  (2 wiersze x 2 kolumny)")

plt.close(fig)  # zamykamy bo nic nie rysujemy — tylko inspekcja typów

**Co właśnie wyłonił print:**

- `Figure` to klasa kontenerowa — trzyma wszystkie panele i metadane figury.
- `axes` zwrócony z `plt.subplots(2,2)` to `numpy.ndarray` (tablica NumPy 2x2) zawierająca 4 obiekty `Axes`.
- Każdy `axes[i,j]` to **pełnoprawny obiekt** z metodami `set_title`, `set_xlabel`, `bar`, `plot`, `scatter`...

**Co Ci to daje praktycznie:**

1. *Dashboard* — `fig` z `GridSpec`, każdy `ax` to inny typ wykresu, ale wszystko jeden plik PNG.
2. *Twin axes* — `ax2 = ax.twinx()` daje drugą oś Y na tym samym `ax` (np. wartość bezwzględna + procent).
3. *Funkcje wielokrotnego użytku* — `def rysuj(ax, data): ax.bar(...)` — wykresy stają się kompozycyjnym budulcem.

Dashboard z 6 paneli to po prostu jeden `Figure` z 6 `Axes`. Dziś zbudujemy taki dashboard od podstaw.

## 2. Seaborn — pierwsze spotkanie

Sprawdźmy intuicję ze szczerością. Spójrz na ten kod i odpowiedz sobie: *„co tu dostanę?"*

In [ ]:
# Pierwszy wykres seaborn — jedna linia
sns.barplot(data=tips, x='day', y='total_bill')
plt.show()
plt.close()

Tak — seaborn zwraca słupki ze średnią + przedziały ufności (CI), z *etykietą osi X automatycznie z nazwy kolumny*, z *gotową paletą*. Trzy „darmowe" rzeczy, których matplotlib nie da z pudełka.

**Skąd seaborn wie, że na osi X mają być dni?** Bo dał mu `data=tips` (cały DataFrame) i `x='day'` (nazwa kolumny). To jest *grammar of graphics* (gramatyka grafiki — koncepcja Leland Wilkinson 2005, rozwinięta w *ggplot2* — bibliotece wizualizacji w języku **R** — przez Hadley Wickhama 2010). **R** = język programowania popularny w statystyce i akademii, alternatywa Pythona w analityce.

## 3. Grammar of Graphics — gramatyka wykresu

Seaborn nie ma „typów wykresów". Ma **gramatykę**:

| Element gramatyki | Argument seaborn | Co robi |
|-------------------|------------------|---------|
| **data** (źródło) | `data=df` | DataFrame z którego rysujemy |
| **aesthetics** (estetyka, mapowanie) | `x=`, `y=`, `hue=`, `style=`, `size=` | które kolumny → na które wymiary wizualne |
| **geom** (kształt) | wybór funkcji: `barplot`/`boxplot`/`scatterplot`... | jaki kształt narysować |
| **stat** (statystyka) | `estimator=`, `errorbar=` | jaką statystykę policzyć (mean, median, count...) |
| **coord** (układ współrzędnych) | `ax.set_xscale('log')`, `ax.set_aspect()` | linear/log, proporcje |
| **theme** (motyw) | `sns.set_theme()`, `palette=`, `sns.axes_style()` | wygląd globalny |

**Konsekwencja:** gdy znasz `sns.barplot(data, x, y, hue)`, znasz też `sns.boxplot(data, x, y, hue)` — to *to samo wywołanie* z innym `geom`. **Jedna gramatyka — dziesiątki wykresów.** *Plotly* (biblioteka interaktywnych wykresów Pythona, JavaScript pod spodem) i *ggplot2* (R) używają tej samej gramatyki — różnica tylko w API (interfejsie programowania).

## ⚠️ Pułapka mentalna 1: „Seaborn zastępuje matplotlib"

Bardzo częsta intuicja po pierwszym kontakcie: *"seaborn jest piękniejszy, więc używam tylko jego, matplotlib nie potrzebuję"*.

**Błąd.** Seaborn to **wrapper** (nakładka) na matplotlib. Sprawdźmy:

In [ ]:
ax = sns.barplot(data=tips, x='day', y='total_bill')
print(f"type(ax)        : {type(ax)}")
print(f"type(ax.figure) : {type(ax.figure)}")
print(f"type(ax) module : {type(ax).__module__}")
plt.close()

**Co właśnie zobaczyłeś:** seaborn zwrócił obiekt klasy `matplotlib.axes._axes.Axes`. To jest *ten sam* obiekt, którego używałeś w W09 z czystym matplotlib.

**Konsekwencja praktyczna:** wszystko, co umiesz z W09 (set_title, set_xlabel, set_ylabel, legend, *twinx* — drugie osie Y, savefig...) **działa po seaborn**:

```python
ax = sns.barplot(data=tips, x='day', y='total_bill')
ax.set_title('Mój tytuł')              # matplotlib API
ax.set_xlabel('Dzień tygodnia')        # matplotlib API
ax.figure.savefig('plik.png', dpi=150) # matplotlib API
```

Seaborn dodaje *nowe funkcje* (barplot, heatmap, pairplot, automatyczne CI), nie zastępuje istniejących. **Profesjonalny workflow:** seaborn dla głównego rysunku, matplotlib do polerowania (tytuły, etykiety, eksport).

## 4. Dwa style API matplotlib — implicit (pyplot) vs explicit (object-oriented)

W matplotlib istnieją dwa style pisania kodu — i **oba można mieszać w tej samej komórce** (skąd nieskończone zdziwienia).

| Styl | Forma | Kiedy używać |
|------|-------|--------------|
| **Implicit** (pyplot, MATLAB-like) | `plt.title()`, `plt.xlabel()`, `plt.bar(x,y)` | Jeden wykres w komórce, szybki one-liner |
| **Explicit** (object-oriented) | `ax.set_title()`, `ax.set_xlabel()`, `ax.bar(x,y)` | Wiele wykresów, dashboard, funkcje wielokrotnego użytku |

**Co dzieje się pod spodem:** `plt.title('A')` to skrót do `plt.gca().set_title('A')` (gca = *get current axes* — pobierz aktualny ax z globalnego stanu pyplot). Czyli `plt` to *state machine* (maszyna stanów) trzymająca „ostatnio aktywny ax".

**Pułapka — typowy *bug* (błąd w kodzie) w pętli — *side effect* (efekt uboczny zmieniający globalny stan):**
```python
fig, axes = plt.subplots(1, 3)
for i, ax in enumerate(axes):
    ax.bar([1,2,3], [i+1, i+2, i+3])
    plt.title(f'Wykres {i}')   # BŁĄD: plt.title trafi w ostatni aktywny ax — globalnie
                               # POPRAWNIE: ax.set_title(f'Wykres {i}')
```

**Konwencja W09 i W10:** używamy *explicit API* (`fig, ax = plt.subplots(); ax.set_...`) — wszystko jest jednoznaczne, brak side effects, kod nadaje się do funkcji.

## 2. Barplot — porównanie wartości między grupami

Pierwszy „prawdziwy" wykres seaborn. Barplot to **najczęściej używany typ wykresu w analityce biznesowej** — porównanie wartości między kategoriami.

### 2.1 Worked example — barplot z grupami (`hue=`)

In [ ]:
# barplot — średnia + 95% CI, podzielona na grupy przez hue=
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Lewy: średni rachunek per dzień, podział wg płci
sns.barplot(
    data=tips,
    x='day',
    y='total_bill',
    hue='sex',
    ax=axes[0],
    palette='muted',
    errorbar=('ci', 95),
)
axes[0].set_title('Średni rachunek wg dnia i płci', fontsize=12)
axes[0].set_xlabel('Dzień tygodnia')
axes[0].set_ylabel('Średni rachunek (USD)')

# Prawy: średni napiwek per dzień, podział wg pory dnia
sns.barplot(
    data=tips,
    x='day',
    y='tip',
    hue='time',
    ax=axes[1],
    palette='Set2',
    errorbar=('ci', 95),
)
axes[1].set_title('Średni napiwek wg dnia i pory', fontsize=12)
axes[1].set_xlabel('Dzień tygodnia')
axes[1].set_ylabel('Średni napiwek (USD)')

fig.suptitle('barplot — automatyczne 95% CI (wąsy na słupkach)', fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

**Co tu się dzieje:**

- `hue='sex'` — *aesthetics mapping* (mapowanie estetyczne): kolumna `sex` → kolor słupka. Seaborn rozdzieli słupki na 2 grupy (Male/Female) i pokoloruje wg palety.
- `errorbar=('ci', 95)` — pokaż 95% CI (przedział ufności średniej). Seaborn wylicza CI metodą *bootstrap* (1000 prób losowych z powtórzeniem) — to jest *stat layer* w grammar of graphics.
- `palette='muted'` / `'Set2'` — *theme layer*: zestaw kolorów. `muted` jest profesjonalna i stonowana.

**Pytanie biznesowe na które odpowiada wykres po lewej:** *Czy mężczyźni i kobiety zostawiają podobne rachunki w różne dni tygodnia?* Odpowiedź widoczna z wykresu: w piątek i niedzielę słupki M/F zachodzą się wąsami → **różnica nieistotna statystycznie**. W sobotę słupek M jest wyraźnie wyższy → **istotna różnica**.

> *Why this:* używamy `barplot` zamiast `plt.bar`, bo *barplot odpowiada na 2 pytania jednocześnie* (średnia + niepewność). `plt.bar` odpowiada na jedno (sama wartość). Barplot z CI to wizualna wersja t-testu (zobaczysz w W11-W12).

## ⚠️ Pułapka mentalna 2: Wąsy w `barplot` to NIE odchylenie standardowe

Bardzo częsty błąd interpretacji: *„wąsy pokazują rozrzut danych"*. Nieprawda.

**Wąsy w `sns.barplot` to przedział ufności średniej (CI mean)**, liczony przez *bootstrap*. Mówią: *„prawdziwa średnia z 95% prawdopodobieństwem leży w tym przedziale"*. **To NIE jest** ani odchylenie standardowe, ani min-max, ani IQR.

**Konsekwencja:** im więcej obserwacji, tym węższy CI — *nawet jeśli rozrzut jest duży*. Mała próbka (n=5) → szeroki CI. Duża próbka (n=100) → wąski CI.

Sprawdźmy to na danych:

In [ ]:
# Sprawdzenie: count vs std vs CI per dzień
stats = tips.groupby('day', observed=True)['total_bill'].agg(['count', 'mean', 'std']).round(2)
print(stats)
print()
print("Interpretacja wąsów barplot:")
print("- Sat (n=87): wąski CI mimo std=9.48 (duża próbka pewnie szacuje średnią)")
print("- Fri (n=19): szeroki CI mimo podobnego std (mała próbka -> niepewność)")
print()
print("Inne opcje errorbar w barplot:")
print("  errorbar=('ci', 95)   # domyslne — przedzial ufnosci 95% (co widzielismy)")
print("  errorbar='sd'         # odchylenie standardowe (rozrzut danych)")
print("  errorbar=('pi', 95)   # przedzial predykcji (PI — gdzie 95% danych)")
print("  errorbar=('se', 1)    # standard error +/- 1 SE (blad standardowy sredniej)")
print("  errorbar=None         # bez wasow")

**Wniosek praktyczny:** zanim zinterpretujesz różnicę w barplocie, **sprawdź `groupby().count()`**. Mała próbka = duża niepewność = nie wyciągaj zbyt mocnych wniosków.

> W zaawansowanej statystyce (W11-W12) zobaczysz formalne testy istotności — *t-test* (test t Studenta, porównanie 2 średnich) i *ANOVA* (analiza wariancji, porównanie >2 średnich). Barplot z CI to ich wizualna wersja dostępna od ręki.

### 5.2 Mini-ćwiczenia — barplot w 3 różnych domenach biznesowych

Trzy zadania o rosnącej trudności: pierwszy przykład robisz z prowadzącym, drugi z lukami do wypełnienia, trzeci samodzielnie.

**Mini-ćwiczenie A — e-commerce:** porównanie konwersji kampanii reklamowych

In [ ]:
# A: e-commerce (handel internetowy) — kampanie reklamowe
# Skróty: email = mailing; fb_ads = reklamy Facebook; google_ads = reklamy Google; sms = SMS
np.random.seed(42)
kampania = np.repeat(['email', 'fb_ads', 'google_ads', 'sms'], 30)
segment = np.tile(np.repeat(['nowi', 'lojalni'], 15), 4)

# Bazowa konwersja per kanał + efekt segmentu (lojalni klienci konwertują +0.6%)
baza = {'email': 2.5, 'fb_ads': 3.8, 'google_ads': 5.2, 'sms': 1.9}
sigma = {'email': 0.4, 'fb_ads': 0.6, 'google_ads': 0.5, 'sms': 0.3}
konwersja = np.array([
    np.random.normal(baza[k] + (0.6 if s == 'lojalni' else 0), sigma[k])
    for k, s in zip(kampania, segment)
])
kampanie = pd.DataFrame({'kampania': kampania, 'segment': segment, 'konwersja_pct': konwersja})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=kampanie, x='kampania', y='konwersja_pct', hue='segment',
            ax=ax, palette='muted', errorbar=('ci', 95))
ax.set_title('Konwersja kampanii reklamowych wg segmentu klienta (%)')
ax.set_xlabel('Kanał')
ax.set_ylabel('Konwersja (%)')
plt.tight_layout()
plt.show()
plt.close()
print(kampanie.groupby(['kampania', 'segment'], observed=True)['konwersja_pct'].mean().round(2))
print("\nWniosek: google_ads dominuje (~5.5%), segment 'lojalni' konwertuje silniej (~+0.6 p.p.)")

**Mini-ćwiczenie B — HR:** średnie wynagrodzenie wg działu i poziomu (uzupełnij brakujące argumenty `<???>`)

> **Twoje zadanie:** uzupełnij parametry oznaczone `<???>`. Reszta kodu (generowanie danych, etykiety, eksport) jest gotowa. Następnie uruchom i odpowiedz: w którym dziale `mid` zarabia najwięcej?

In [ ]:
# B: HR (Human Resources — kadry, dzial personalny) — wynagrodzenia
# Generator danych (zostaw bez zmian):
N = 30
dzialy = ['IT', 'Sales', 'HR', 'Finance']
baza_pensji = {'IT': 11000, 'Sales': 9500, 'HR': 8000, 'Finance': 10500}
mnoznik_poziomu = {'junior': 0.75, 'mid': 1.00, 'senior': 1.40}

rng = np.random.default_rng(7)
rows = []
for dz in dzialy:
    poziomy = rng.choice(['junior', 'mid', 'senior'], size=N, p=[0.25, 0.50, 0.25])
    for p in poziomy:
        pensja = rng.normal(baza_pensji[dz] * mnoznik_poziomu[p], 700)
        rows.append({'dzial': dz, 'poziom': p, 'pensja': pensja})
hr = pd.DataFrame(rows)

# ============================================================================
# TWOJE ZADANIE (faded — czesciowo wypelnione):
# Ponizej ZAKOMENTOWANY szablon ma 4 placeholdery <???>.
# 1. Spojrz na zakomentowany szablon i SAM uzupelnij go w glowie / na kartce.
# 2. Odkryj DZIALAJACY kod (ponizej "ROZWIAZANIE") i porownaj z Twoim.
# 3. Uruchom komorke — porownaj swoj wynik z DataFrame agregacji.
# ============================================================================

# --- SZABLON FADED (zakomentowany — Twoje zadanie: uzupelnic <???>) ---
# fig, ax = plt.subplots(figsize=(9, 5))
# sns.barplot(
#     data=<???>,     # <???> = DataFrame (dostepne: hr)
#     x=<???>,        # <???> = kolumna na os X (kategoria glowna)
#     y=<???>,        # <???> = kolumna na os Y (wartosc liczbowa)
#     hue=<???>,      # <???> = kolumna na podzial kolorystyczny
#     ax=ax, palette='Set2', errorbar=('ci', 95),
#     hue_order=['junior', 'mid', 'senior'],
# )

# --- ROZWIAZANIE (uruchamiane) ---
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=hr, x='dzial', y='pensja', hue='poziom',
    ax=ax, palette='Set2', errorbar=('ci', 95),
    hue_order=['junior', 'mid', 'senior'],
)
ax.set_title('Srednia pensja wg dzialu i poziomu (PLN)')
ax.set_xlabel('Dzial')
ax.set_ylabel('Pensja (PLN)')
plt.tight_layout()
plt.show()
plt.close()
print(hr.groupby(['dzial', 'poziom'], observed=True)['pensja'].agg(['count', 'mean']).round(0))

**Mini-ćwiczenie C — marketing:** ROI kanałów wg kwartału (samodzielnie, bez podpowiedzi)

> Skróty marketingowe: **ROI** (Return On Investment — zwrot z inwestycji, %); **SEO** (Search Engine Optimization — pozycjonowanie w wynikach wyszukiwania); **PPC** (Pay-Per-Click — reklamy płatne za kliknięcie); **content** (marketing treści — blog, video); **social** (media społecznościowe); **PR** (Public Relations — relacje z mediami).
>
> **Twoje zadanie (3 kroki, samodzielnie):**
> 1. Uruchom kod poniżej — przeanalizuj wykres.
> 2. Spójrz na wydruk średnich i wąsy CI — który kanał ma **statystycznie istotnie najwyższy ROI** w Q4 (pamiętaj: jeśli wąsy się zachodzą, różnica nie jest istotna)?
> 3. Linia `break-even` (próg rentowności, ROI=100%) — które kanały są poniżej w którym kwartale?
>
> **Oczekiwany wynik:** w komentarzu pod komórką napisz odpowiedź na pytania 2-3 (3-4 zdania).

In [ ]:
# C: marketing — ROI kanałów per kwartał
rng = np.random.default_rng(99)
kanaly_baza = {'SEO': 180, 'PPC': 120, 'social': 95, 'content': 220, 'PR': 60}
kanaly_sigma = {'SEO': 30, 'PPC': 25, 'social': 35, 'content': 40, 'PR': 20}
# Trend: ROI rośnie w Q4 (sezon zakupowy) o ~15%
trend_kw = {'Q1': 1.0, 'Q2': 1.05, 'Q3': 1.0, 'Q4': 1.15}

rows = []
for kanal in kanaly_baza:
    for kw in trend_kw:
        for _ in range(8):
            v = rng.normal(kanaly_baza[kanal] * trend_kw[kw], kanaly_sigma[kanal])
            rows.append({'kanal': kanal, 'kwartal': kw, 'roi_pct': v})
roi = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=roi, x='kanal', y='roi_pct', hue='kwartal',
            ax=ax, palette='colorblind', errorbar=('ci', 95))
ax.set_title('ROI kanałów marketingowych wg kwartału')
ax.set_xlabel('Kanał')
ax.set_ylabel('ROI (%)')
ax.axhline(100, color='red', linestyle='--', alpha=0.5,
           label='break-even (próg rentowności)')
ax.legend(title='Kwartał', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
plt.close()

# Pomocnicza tabela liczb (do interpretacji):
print(roi.groupby(['kanal', 'kwartal'], observed=True)['roi_pct'].mean().round(1).unstack())

**Trzy domeny — ten sam wzorzec.** Zwróć uwagę: API się nie zmienia. Zmienia się tylko *kolumna* w `x=`/`y=`/`hue=`. To jest moc grammar of graphics: jednej składni do wszystkich problemów biznesowych.

## 3. Boxplot i Violinplot — rozkład danych, nie tylko średnia

Barplot pokazał *średnią*. Ale często ważniejsze jest *jak dane są rozłożone*. Czy mediana jest blisko średniej? Czy są wartości odstające (*outliery*)? Czy rozkład ma jeden szczyt czy dwa?

Na to są **boxplot** (skrzynka z wąsami) i **violinplot** (rozkład gęstości).

### 3.1 Boxplot — Q1, mediana, Q3, IQR, *outliery* (wartości odstające)

> **Powtórka z W09:** *percentyl* (wartość poniżej której znajduje się dany % obserwacji): Q1 (pierwszy kwartyl) = 25. percentyl, Q2 = mediana = 50. percentyl, Q3 = 75. percentyl. **IQR** (*interquartile range* — rozstęp międzykwartylowy) = Q3 − Q1.

In [ ]:
# boxplot — pełny rozkład jednym wykresem
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Lewy: rachunki per dzień, podział lunch/dinner
sns.boxplot(
    data=tips,
    x='day',
    y='total_bill',
    hue='time',
    ax=axes[0],
    palette='pastel',
)
axes[0].set_title('Rozkład rachunków wg dnia (Lunch vs Kolacja)', fontsize=11)
axes[0].set_xlabel('Dzień tygodnia')
axes[0].set_ylabel('Rachunek (USD)')

# Prawy: napiwki per dzień (hue='day' + legend=False — wymóg seaborn 0.13+ przy palette=)
sns.boxplot(
    data=tips,
    x='day',
    y='tip',
    hue='day',
    legend=False,
    ax=axes[1],
    palette='Set3',
)
axes[1].set_title('Rozkład napiwków wg dnia', fontsize=11)
axes[1].set_xlabel('Dzień tygodnia')
axes[1].set_ylabel('Napiwek (USD)')

fig.suptitle('boxplot — mediana, IQR (rozstęp międzykwartylowy) i wartości odstające',
             fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

**Anatomia boxplota:**

```
       *     <- outlier (wartość odstająca, > Q3 + 1.5*IQR)
       |
    ───┬───  <- górny wąs (whisker, max do Q3 + 1.5*IQR)
    │     │
    │  ─  │  <- Q3 (75. percentyl)
    │     │
    │  ─  │  <- mediana (Q2, 50. percentyl)
    │     │
    │  ─  │  <- Q1 (25. percentyl)
    │     │
    ───┴───  <- dolny wąs
```

**Po co boxplot zamiast średniej?** Bo **mediana jest odporna na outliery** (jeden klient na 1000 USD nie zniekształci). Średnia — nie. Boxplot pokazuje 5 statystyk + outliery na jednym wykresie.

**Pytanie biznesowe:** *„W którym dniu klienci zostawiają najbardziej zróżnicowane napiwki?"*. Z prawego wykresu: **sobota** ma najszerszą skrzynkę (duża rozpiętość Q1-Q3) i najwięcej *outlierów* (wartości odstających, daleko od typowego rozkładu) — niektórzy klienci dają duże napiwki, inni minimalne.

### 3.2 Violinplot — boxplot + KDE (*kernel density estimation* — wygładzona estymacja gęstości)

In [ ]:
# violinplot — rozklad jako gestosc prawdopodobienstwa (KDE — kernel density estimation,
# wygladzona estymacja gestosci; mowi gdzie wartosci sa zageszczone)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Lewy: rachunki Lunch vs Kolacja, split wg plci
sns.violinplot(
    data=tips,
    x='time',
    y='total_bill',
    hue='sex',
    split=True,         # split=True wymaga dokladnie 2 kategorii w hue
    ax=axes[0],
    palette='muted',
    inner='box',        # wewnatrz pokaz mini-boxplot
)
axes[0].set_title('Rachunki Lunch vs Kolacja (podział wg płci)', fontsize=11)
axes[0].set_xlabel('Pora dnia')
axes[0].set_ylabel('Rachunek (USD)')

# Prawy: napiwki per dzien (hue='day' + legend=False — wymog seaborn 0.13+)
sns.violinplot(
    data=tips,
    x='day',
    y='tip',
    hue='day',
    legend=False,
    ax=axes[1],
    palette='Set2',
    inner='box',
)
axes[1].set_title('Rozkład napiwków wg dnia', fontsize=11)
axes[1].set_xlabel('Dzień tygodnia')
axes[1].set_ylabel('Napiwek (USD)')

fig.suptitle('violinplot — kształt skrzypiec = gęstość danych (KDE)', fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

**Co dodaje violinplot do boxplota:**

- **Kształt** skrzypiec to *KDE* (kernel density estimation — wygładzona estymacja gęstości). Tam gdzie skrzypce są szerokie — dużo obserwacji o tej wartości. Wąskie — mało.
- **`split=True` z `hue`** — dwie połówki skrzypiec obok siebie (np. lewa = Male, prawa = Female). Genialne do porównań.
- **`inner='box'`** — wewnątrz każdej skrzypce mini-boxplot (mediana + IQR).

**Kiedy violinplot bije boxplota?**

1. **Rozkład wielomodalny** (z więcej niż jednym szczytem — np. dwa skupiska wartości) — boxplot to ukryje, violinplot pokaże dwa wybrzuszenia.
2. **Bardzo skośny rozkład** (asymetryczny — większość wartości po jednej stronie) — kształt skrzypiec mówi o asymetrii lepiej niż wąsy.

## 4. Heatmap — gdy dane są dwuwymiarowe

### 4.1 Macierz korelacji + tabela przestawna

In [ ]:
# heatmap — dwa najczęstsze zastosowania: korelacja i pivot table
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# LEWY: korelacja zmiennych numerycznych
corr = tips.select_dtypes('number').corr()
sns.heatmap(
    corr,
    annot=True,        # liczby w komórkach
    fmt='.2f',         # 2 miejsca po przecinku
    cmap='coolwarm',   # paleta dywergencyjna: czerwony=+, niebieski=-
    center=0,          # 0 = biały (środek dywergencji)
    ax=axes[0],
    square=True,
    cbar_kws={'label': 'współczynnik korelacji'},
)
axes[0].set_title('Macierz korelacji zmiennych numerycznych', fontsize=11)

# PRAWY: pivot heatmap — średni rachunek per dzień i pora
pivot = tips.pivot_table(
    values='total_bill',
    index='day',
    columns='time',
    aggfunc='mean',
    observed=True,    # observed=True dla kategorii — usuwa FutureWarning
)
sns.heatmap(
    pivot,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',    # paleta sekwencyjna: jasne -> ciemne
    ax=axes[1],
    cbar_kws={'label': 'średni rachunek (USD)'},
)
axes[1].set_title('Pivot table: średni rachunek (dzień × pora)', fontsize=11)

fig.suptitle('heatmap — korelacja (dywergencyjna) i pivot (sekwencyjna)', fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

**Dwa zupełnie różne zastosowania heatmapy:**

| Zastosowanie | Dane wejściowe | Paleta | Co pokazuje |
|--------------|----------------|--------|-------------|
| **Korelacja** | macierz NxN (`df.corr()`) | dywergencyjna (coolwarm, RdBu_r) | siłę i kierunek związku między zmiennymi |
| **Pivot table** | tabela 2-wymiarowa (`pivot_table`) | sekwencyjna (YlOrRd, viridis, Blues) | wartość zagregowaną w 2 wymiarach kategorycznych |

**Kluczowa decyzja: paleta dywergencyjna vs sekwencyjna**

- **Dywergencyjna** (`coolwarm`, `RdBu_r`, `PiYG`) — gdy 0 lub środkowa wartość ma znaczenie (np. korelacja: -1...0...+1, profit/loss).
- **Sekwencyjna** (`Blues`, `viridis`, `YlOrRd`) — gdy skala idzie od minimum do maksimum bez „środka" (np. wartości pieniędzy, wieku, liczności).

## ⚠️ Pułapka mentalna 3: „Heatmapa to tabela z kolorami"

Częste myślenie: *„heatmap to ładniejszy Excel — pokazuje liczby z kolorem"*. **Błąd.**

Heatmapa to **encoded matrix** — kolor reprezentuje **trzeci wymiar** (wartość liczbową), gdzie pozycja (wiersz, kolumna) niesie pierwsze dwa wymiary. Wszystkie trzy razem tworzą informację. Bez koloru pozostałaby zwykła tabela.

**Kiedy heatmapa jest dobrym wyborem? Reguła kciuka:**

- ✅ **>10 wierszy × >5 kolumn** — kolor pomaga skanować wzrokiem; tabela byłaby nieczytelna.
- ⚠️ **5-10 × 3-5** — granica; pomaga jeśli wartości są zróżnicowane.
- ❌ **<5 × <3** — heatmapa to ozdoba, lepsza tabela liczb lub barplot.

**Antywzorzec do pokazania (poniższy kod uruchom i porównaj z prawym wykresem powyżej):**

In [ ]:
# ANTYWZORZEC: heatmapa zbyt mała (3x4) — lepsza byłaby tabela
small = tips.pivot_table(
    values='tip', index='time', columns='sex', aggfunc='mean', observed=True,
)
fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(small, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Antywzorzec: heatmapa 2x2 — przerost formy nad treścią')
plt.tight_layout()
plt.show()
plt.close()
print("Lepiej:")
print(small.round(2).to_string())
print("\n4 liczby — nie potrzebują koloru. Heatmapa staje się ozdobnikiem.")

## ⚠️ Pułapka mentalna 4: „Im więcej kolorów, tym lepszy wykres"

Edward Tufte (*The Visual Display of Quantitative Information*, 1983/2001) wprowadził pojęcie ***data-ink ratio*** — każdy piksel powinien nieść informację.

Kolory niosą znaczenie tylko gdy:
- Odpowiadają **kategorii** (palety jakościowe — `muted`, `Set2`, `colorblind`)
- Odpowiadają **skali liniowej** (palety sekwencyjne — `Blues`, `viridis`, `YlOrRd`)
- Odpowiadają **dywergencji wokół 0** (palety dywergencyjne — `coolwarm`, `RdBu_r`)

**Tęczowe palety** (`jet`, `rainbow`) są **percepcyjnie nieliniowe** — sztucznie podkreślają niektóre wartości. Były usunięte jako domyślne z matplotlib 2.0 (2017).

**Historia matplotlib viridis:** w 2015 (matplotlib 1.5) Stéfan van der Walt i Nathaniel Smith **dodali** paletę *viridis* (zielono-żółto-fioletowa, percepcyjnie liniowa). Matematycznie zaprojektowana, by była: (a) percepcyjnie liniowa (równe skoki wartości = równe skoki postrzeganej jasności), (b) widoczna dla osób z zaburzeniami widzenia kolorów, (c) działała na wydruku czarno-białym. **Stała się paletą domyślną w matplotlib 2.0 (2017)**, zastępując starą `jet`. Dziś jest standardem w *Nature*, *Science* i większości czasopism naukowych.

In [ ]:
# Antywzorzec vs wzorzec: ta sama korelacja, dwie palety
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# LEWY: rainbow — antywzorzec
sns.heatmap(corr, annot=True, fmt='.2f', cmap='rainbow', ax=axes[0], center=0)
axes[0].set_title("ANTYWZORZEC: cmap='rainbow'\n(korelacja 0 nie wyróżniona kolorem)",
                  fontsize=11, color='red')

# PRAWY: coolwarm — wzorzec
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], center=0)
axes[1].set_title("WZORZEC: cmap='coolwarm'\n(0 = biały, +/- = niebieski/czerwony)",
                  fontsize=11, color='green')

fig.suptitle('Paleta dywergencyjna pokazuje znak korelacji, tęczowa go ukrywa',
             fontsize=13)
plt.tight_layout()
plt.show()
plt.close()

**Wniosek:** dla danych liczbowych zawsze wybieraj paletę adekwatną do struktury danych. Tęczowa NIGDY nie jest poprawnym wyborem dla danych liczbowych.

> **Ciekawostka — Hans Rosling i Gapminder** *(pamiętasz z W09 sekcji „Polecane do dalszej nauki"? Tutaj wracamy do niego z innej strony — palet kolorów)*. Animacje *Gapminder* używały palety sekwencyjnej (rozmiar bąbla = populacja, oś X = PKB, oś Y = średnia długość życia) — gdyby Rosling użył palety tęczowej, jego wykład TED 2006 by się nie udał. Wybór palety nie jest stylem — jest argumentem.

## 5. Pairplot (wykres macierzy par) — eksploracyjna analiza danych (EDA)

> **EDA** (*Exploratory Data Analysis* — eksploracyjna analiza danych) to faza przed właściwym modelowaniem: oglądasz dane, szukasz wzorców, anomalii, korelacji. **Pairplot** to *narzędzie #1 EDA*.

In [ ]:
# pairplot — macierz wszystkich par zmiennych + rozkłady na przekątnej
# UWAGA: pairplot tworzy WŁASNĄ figurę i NIE przyjmuje ax= (więcej niżej)
g = sns.pairplot(
    tips[['total_bill', 'tip', 'size', 'sex']],
    hue='sex',
    diag_kind='kde',         # rozkłady na przekątnej jako KDE
    plot_kws={'alpha': 0.6}, # przezroczystość punktów
)
g.fig.suptitle('Pairplot — wszystkie zmienne numeryczne vs płeć',
               y=1.02, fontsize=13)
plt.show()
plt.close()

**Co dostajesz w jednej linii kodu:**

- Macierz N×N wykresów (gdzie N = liczba zmiennych numerycznych)
- **Off-diagonal** (poza przekątną): *scatter* (wykres rozrzutu) każdej pary zmiennych
- **Diagonal** (przekątna): rozkład każdej zmiennej osobno (KDE lub histogram)
- **`hue=`**: koloruje wszystko wg kategorii — natychmiastowe „czy płeć ma znaczenie?"

**Pytanie biznesowe:** *"Czy istnieją zmienne silnie ze sobą skorelowane?"* Z wykresu: total_bill vs tip → wyraźna korelacja (chmura punktów wzdłuż linii). total_bill vs size → też. tip vs size → słabsza ale widoczna. *Pairplot odpowiedział na pytanie w 5 sekund*.

## ⚠️ Pułapka mentalna 5: `pairplot` w siatce subplotów

Częsta próba: *„chcę wstawić pairplot do dashboardu — `sns.pairplot(data, ax=axes[0,0])`"*. **Nie zadziała.**

In [ ]:
# Sprawdzamy: pairplot zwraca PairGrid, nie pojedynczy Axes
g = sns.pairplot(tips[['total_bill', 'tip']], height=2)
print(f"type(g)        : {type(g).__name__}")
print(f"type(g.fig)    : {type(g.fig).__name__}")
print(f"g.axes shape   : {g.axes.shape}  (2x2 macierz Axes)")
plt.close('all')

**Dlaczego nie da się?** `pairplot` zwraca obiekt `seaborn.axisgrid.PairGrid` — to *kolekcja N×N axes* na własnej Figure. Tworzy własną figurę, **nie przyjmuje `ax=`**.

To samo dotyczy innych funkcji *figure-level* w seaborn:

| Funkcja | Typ | Przyjmuje `ax=`? | W subplots? |
|---------|-----|-------------------|-------------|
| `barplot`, `boxplot`, `violinplot`, `stripplot` | axes-level | ✅ TAK | ✅ TAK |
| `heatmap`, `scatterplot`, `lineplot`, `histplot` | axes-level | ✅ TAK | ✅ TAK |
| **`pairplot`** | figure-level | ❌ NIE | ❌ NIE |
| **`jointplot`** | figure-level | ❌ NIE | ❌ NIE |
| **`catplot`** | figure-level | ❌ NIE | ❌ NIE |
| **`relplot`**, **`displot`**, **`lmplot`** | figure-level | ❌ NIE | ❌ NIE |

**Reguła kciuka:** funkcje *figure-level* (poziom-figury) — kończące na `-plot` BEZ konkretnego *geom* (jointplot, catplot, relplot, displot, lmplot, pairplot) → osobna figura, nie do dashboardu. Funkcje *axes-level* (poziom-osi) — barplot, boxplot, scatterplot, heatmap — przyjmują `ax=` i działają w siatce.

## 6. Style i palety — wybór profesjonalisty

W sekcji 4 widzieliśmy już palety. Tutaj pełen przegląd stylów oraz zasad wyboru palet.

### 6.1 Style seaborn (4 opcje)

In [ ]:
# Porownanie 4 stylow — context manager (menedzer kontekstu) sns.axes_style()
# Context manager (with-blok) tymczasowo zmienia styl, po wyjsciu wraca do globalnego.
styles = ['whitegrid', 'darkgrid', 'white', 'ticks']
fig, axes = plt.subplots(1, 4, figsize=(16, 3), constrained_layout=True)

for ax, style in zip(axes, styles):
    with sns.axes_style(style):
        sns.barplot(data=tips, x='day', y='total_bill', hue='day', legend=False,
                    ax=ax, palette='muted')
        ax.set_title(f"style='{style}'", fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('Rachunek' if ax is axes[0] else '')

fig.suptitle('Porównanie stylów seaborn (context manager)', fontsize=13)
plt.show()
plt.close()

**Kiedy który styl:**
- `whitegrid` — prezentacje (czytelna siatka pomaga odczytać wartości)
- `white` — publikacje naukowe (minimalistyczne, jak *Nature*)
- `ticks` — raporty techniczne (znaczniki na osiach)
- `darkgrid` — dashboardy interaktywne, ciemny motyw

**Context manager `with sns.axes_style(...)`** zmienia styl *tylko w bloku `with`*. Po wyjściu wraca do globalnego ustawienia z `sns.set_theme()`. Dzięki temu możesz zrobić jeden wykres w innym stylu bez psucia reszty notebooka.

### 6.2 Palety — 6 typów do zapamiętania

In [ ]:
# Palety — przeglad 6 najwazniejszych palet jakosciowych (do kategorii)
palettes = ['muted', 'bright', 'pastel', 'deep', 'Set2', 'colorblind']
fig, axes = plt.subplots(2, 3, figsize=(14, 6), constrained_layout=True)

for ax, palette in zip(axes.flat, palettes):
    sns.barplot(data=tips, x='day', y='total_bill', hue='day', legend=False,
                ax=ax, palette=palette)
    ax.set_title(f"palette='{palette}'", fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('Palety jakościowe seaborn — wybór do kategorii', fontsize=13)
plt.show()
plt.close()

**Reguły wyboru palety:**

1. **Dane jakościowe** (kategorie nieuporządkowane: dni tygodnia, departamenty, kanały marketingowe) → `muted`/`Set2`/`colorblind`/`deep`
2. **Dane sekwencyjne** (skala 0 → max: wartości, ilości) → `Blues`/`viridis`/`YlOrRd`/`Greens`
3. **Dane dywergencyjne** (-max → 0 → +max: korelacja, profit/loss) → `coolwarm`/`RdBu_r`/`PiYG`
4. **Materiały publiczne** (raporty, slajdy) → ZAWSZE `colorblind` (8% mężczyzn ma daltonizm)

> **Pułapka:** seaborn 0.13+ rzuca `FutureWarning: passing palette without hue` — od tej wersji `palette=` wymaga jednoczesnego `hue=`. *Workaround* (obejście problemu): jawnie podaj `hue=x` lub ustaw `palette=` jako listę kolorów.

## 7. Zaawansowany matplotlib — układy wykresów

Do tej pory każdy wykres miał *jeden* panel. Profesjonalny raport ma *wiele* paneli na jednej figurze. Trzy techniki:

1. **`plt.subplots(rows, cols)`** — regularna siatka (wszystkie panele równe)
2. **`GridSpec(rows, cols)`** — nieregularna siatka (panele różnej wielkości)
3. **`sharex` / `sharey`** — wspólne osie (lepsze porównanie)

### 7.1 plt.subplots — regularna siatka

In [ ]:
# Siatka 2x3 — 6 wykresów regularnych
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

# Dostęp przez axes[wiersz, kolumna]
sns.barplot(data=tips, x='day', y='total_bill', hue='day', legend=False,
            ax=axes[0, 0], palette='muted')
axes[0, 0].set_title('Barplot — średni rachunek')
axes[0, 0].set_xlabel('Dzień')
axes[0, 0].set_ylabel('Rachunek (USD)')

sns.boxplot(data=tips, x='day', y='tip', hue='day', legend=False,
            ax=axes[0, 1], palette='pastel')
axes[0, 1].set_title('Boxplot — napiwki')
axes[0, 1].set_xlabel('Dzień')

sns.violinplot(data=tips, x='time', y='total_bill', hue='time', legend=False,
               ax=axes[0, 2], palette='Set2')
axes[0, 2].set_title('Violinplot — rachunki')
axes[0, 2].set_xlabel('Pora dnia')

corr = tips.select_dtypes('number').corr()  # lokalnie dla niezaleznosci komorki
sns.heatmap(corr, annot=True, fmt='.2f', ax=axes[1, 0], cbar=False, cmap='coolwarm')
axes[1, 0].set_title('Heatmap — korelacja')

sns.scatterplot(data=tips, x='total_bill', y='tip', hue='day',
                ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title('Scatter — rachunek vs napiwek')

sns.countplot(data=tips, x='day', hue='smoker', ax=axes[1, 2], palette='Set3')
axes[1, 2].set_title('Countplot — liczba wizyt')

fig.suptitle('Siatka 2×3 — plt.subplots(2, 3)', fontsize=15, fontweight='bold')
plt.show()
plt.close()

**Klucz `constrained_layout=True`** — automatycznie dopasowuje marginesy między panelami. To nowsza i lepsza wersja `tight_layout()`. **Domyślnie używaj jej** w każdym `plt.subplots()`.

## ⚠️ Pułapka mentalna 6: „GridSpec to ozdobnik"

Częsta myśl: *„skoro mam `subplots()`, po co mi GridSpec?"*. **Bo profesjonalny dashboard ma hierarchię ważności.**

`subplots(2, 3)` daje 6 paneli identycznej wagi. Ale dashboard analityczny zwykle ma:
- **1 panel główny** (*KPI* — *Key Performance Indicator*, kluczowy wskaźnik efektywności biznesu — np. przychód, konwersja) — duży
- **3-5 paneli pomocniczych** (detale, *breakdown* — rozbicie szczegółowe na komponenty) — mniejsze
- **Czasem 1 panel sumaryczny** (*footer* — stopka, suma) — pełna szerokość

To wymaga `GridSpec`.

### 7.2 GridSpec — nieregularna siatka

In [ ]:
# GridSpec — elastyczna siatka 3x3 z panelami różnej wielkości
fig = plt.figure(figsize=(15, 9), constrained_layout=True)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)

# Górny panel — pełna szerokość (gs[0, :] = wiersz 0, wszystkie kolumny)
ax_top = fig.add_subplot(gs[0, :2])
sns.barplot(
    data=tips, x='day', y='total_bill', hue='sex',
    ax=ax_top, palette='muted', errorbar=('ci', 95),
)
ax_top.set_title('Średni rachunek wg dnia i płci (panel główny — 2/3 szerokości)',
                 fontsize=11)
ax_top.set_xlabel('Dzień tygodnia')
ax_top.set_ylabel('Rachunek (USD)')

# Górny prawy — pojedyncza komórka
ax_tr = fig.add_subplot(gs[0, 2])
day_counts = tips['day'].value_counts()
ax_tr.pie(
    day_counts.values, labels=day_counts.index, autopct='%1.0f%%',
    colors=sns.color_palette('muted')[:len(day_counts)], startangle=90,
)
ax_tr.set_title('Udział dni', fontsize=11)

# Środkowy rząd — trzy równe komórki
ax_m1 = fig.add_subplot(gs[1, 0])
ax_m2 = fig.add_subplot(gs[1, 1])
ax_m3 = fig.add_subplot(gs[1, 2])

sns.boxplot(data=tips, x='time', y='tip', hue='time', legend=False,
            ax=ax_m1, palette='pastel')
ax_m1.set_title('Napiwki lunch/kolacja', fontsize=10)

corr = tips.select_dtypes('number').corr()  # lokalnie dla niezaleznosci komorki
sns.heatmap(corr, annot=True, fmt='.2f', ax=ax_m2, cbar=False, cmap='coolwarm')
ax_m2.set_title('Korelacja zmiennych', fontsize=10)

sns.scatterplot(data=tips, x='total_bill', y='tip', hue='smoker',
                ax=ax_m3, alpha=0.7)
ax_m3.set_title('Rachunek vs napiwek', fontsize=10)

# Dolny — pełna szerokość (gs[2, :])
ax_bot = fig.add_subplot(gs[2, :])
tips_sum = tips.groupby('day', observed=True)['total_bill'].sum().reset_index()
ax_bot.bar(
    tips_sum['day'].astype(str), tips_sum['total_bill'],
    color=sns.color_palette('muted')[:4],
)
ax_bot.set_title('Suma rachunków wg dnia tygodnia (dolny panel — pełna szerokość)',
                 fontsize=11)
ax_bot.set_xlabel('Dzień')
ax_bot.set_ylabel('Suma rachunków (USD)')

fig.suptitle('GridSpec 3×3 — panele różnej wielkości', fontsize=14, fontweight='bold')
plt.show()
plt.close()

**Składnia `GridSpec`:**

```python
gs = gridspec.GridSpec(3, 3, figure=fig)   # 3 wiersze x 3 kolumny — łącznie 9 komórek

ax = fig.add_subplot(gs[0, :])     # cały pierwszy wiersz   (1x3 komórki)
ax = fig.add_subplot(gs[0, :2])    # pierwszy wiersz, 2 kolumny (1x2)
ax = fig.add_subplot(gs[1, 0])     # konkretna komórka (1x1)
ax = fig.add_subplot(gs[1:3, 0])   # 2 wiersze, 1 kolumna (2x1)
ax = fig.add_subplot(gs[2, :])     # cały trzeci wiersz (1x3)
```

To jest **slice notation** (notacja wycinka) jak w NumPy — przyjazne dla każdego, kto zna pandas/numpy.

> **Why this, why now, why not X:** używamy `GridSpec` zamiast `subplots()`, gdy panele mają **różne rozmiary** (raport zarządczy: 1 duży KPI + 4 małe wykresy pomocnicze). W tym momencie kursu (po W09 z prostym `subplots`) `GridSpec` to naturalna ewolucja. Alternatywa `subplot_mosaic()` (matplotlib 3.4+) daje podobne efekty z czytelniejszą składnią ASCII — pokażemy ją krótko jako *nice-to-know*, ale `GridSpec` jest standardem w 90% kodu w środowisku produkcyjnym.

### 7.3 Powtórka z W08 — `groupby` w roli przygotowania danych

Wracamy do `groupby` z W08 — tym razem jako fundament każdego wykresu agregowanego.

In [ ]:
# Powtórka z W08: groupby + agg() jako fundament każdego dashboardu
agg_tabela = tips.groupby('day', observed=True).agg(
    rachunki=('total_bill', 'sum'),
    sredni=('total_bill', 'mean'),
    napiwki=('tip', 'sum'),
    wizyty=('total_bill', 'count'),
).round(2)
print("Tabela agregowana (z W08) — każdy wiersz to dzień:")
print(agg_tabela)
print()
print("Te liczby zaraz pokażemy jako 4 panele dashboardu — to jest most")
print("między W08 (Pandas merge/groupby) a W10 (wizualizacja).")

**Wniosek metapoznawczy:** wizualizacja **nie istnieje bez przygotowania danych**. Każdy panel dashboardu to jakaś agregacja: `sum()`, `mean()`, `count()`, `pivot_table()`. Bez W08 (Pandas) wizualizacja nie ma sensu — bez W10 (wizualizacja) agregacje są nieczytelne.

## 8. Long vs Wide format danych — przygotowanie pod dashboard

Zanim zbudujemy dashboard z 6 paneli, jedna jeszcze rzecz. Bez niej seaborn będzie się Ci „dziwnie zachowywał" przy każdej tabeli z Excela.

**Tidy data** (uporządkowane dane — standard analityczny rozpowszechniony przez Hadley Wickhama, twórcę pakietu *tidyverse* w R):
- Każda **zmienna** = osobna **kolumna**
- Każda **obserwacja** = osobny **wiersz**
- Każdy **typ obserwacji** = osobna **tabela**

To jest **long format**. Seaborn został zaprojektowany pod long format. Excel-importy są zwykle wide.

In [ ]:
# Wide format — typowy import z Excela
sales_wide = pd.DataFrame({
    'sklep':       ['A', 'B', 'C'],
    'poniedzialek': [100, 120, 90],
    'wtorek':      [110, 130, 95],
    'sroda':       [105, 125, 100],
})
print("WIDE format (sklepy w wierszach, dni w kolumnach):")
print(sales_wide)
print()

# Konwersja wide -> long przez melt()
sales_long = sales_wide.melt(
    id_vars='sklep',
    var_name='dzien',
    value_name='sprzedaz',
)
print("LONG format (po melt — każda obserwacja w osobnym wierszu):")
print(sales_long)

**Diagram konwersji:**

```
WIDE (3x4)                          LONG (9x3)
┌──────┬─────┬─────┬─────┐          ┌──────┬─────┬─────────┐
│sklep │pn   │wt   │sr   │          │sklep │dzien│sprzedaz │
├──────┼─────┼─────┼─────┤  melt()  ├──────┼─────┼─────────┤
│ A    │100  │110  │105  │ ───────► │ A    │pn   │100      │
│ B    │120  │130  │125  │          │ A    │wt   │110      │
│ C    │ 90  │ 95  │100  │ ◄─────── │ A    │sr   │105      │
└──────┴─────┴─────┴─────┘  pivot() │ B    │pn   │120      │
                                    │ B    │wt   │130      │
                                    │ ...  │ ... │ ...     │
                                    └──────┴─────┴─────────┘
        sns.barplot(data=long, x='dzien', y='sprzedaz', hue='sklep')
        # 1 linia, 3 sklepy w 3 kolorach, 3 dni na osi X
```

In [ ]:
# Wykres z long format — jedna linia, gotowe
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=sales_long, x='dzien', y='sprzedaz', hue='sklep',
            ax=ax, palette='muted')
ax.set_title('Sprzedaż per dzień i sklep (long format → 1 linia kodu)')
ax.set_xlabel('Dzień tygodnia')
ax.set_ylabel('Sprzedaż (zł)')
plt.tight_layout()
plt.show()
plt.close()

**Reguła praktyczna:**

| Co chcesz zrobić? | Format | Funkcja konwersji |
|-------------------|--------|-------------------|
| Wykres seaborn z `hue=` | **long** | `df.melt()` |
| Heatmapa (matrix) | **wide** | `df.pivot_table()` |
| Tabele do Excela / raporty | **wide** | `df.pivot_table()` |
| Modelowanie scikit-learn | **long** zwykle | `df.melt()` |

**Why this, why now, why not X:** używamy long format bo seaborn `hue=` wymaga go natywnie. Bez konwersji długość kodu rośnie 5×: musiałbyś pętlą iterować po kolumnach i robić ręczne `bar()` z manualnym przesunięciem X. Long + seaborn = jedna linia.

## 🤔 Sprawdź siebie

> **Stop. Nie patrz dalej.** Spróbuj odpowiedzieć **bez** patrzenia w komórki powyżej. To pytanie sprawdza, czy idea grammar of graphics weszła w intuicję.

**Pytanie:** Otrzymujesz DataFrame `pacjenci` z kolumnami: `oddzial` (kategoria), `wiek` (liczba), `czas_pobytu_dni` (liczba), `plec` (kategoria). Chcesz pokazać: *„czy mężczyźni i kobiety mają różny rozkład czasu pobytu w różnych oddziałach?"*. **Który kod robi to najprościej?**

**(A)** `plt.bar(pacjenci['oddzial'], pacjenci['czas_pobytu_dni'])`

**(B)** `sns.boxplot(data=pacjenci, x='oddzial', y='czas_pobytu_dni', hue='plec')`

**(C)** `sns.heatmap(pacjenci.corr())`

**(D)** `sns.pairplot(pacjenci, hue='plec')`

> Daj sobie 30 sekund, **wybierz odpowiedź**, potem spójrz niżej.

<details>
<summary>📖 Kliknij aby zobaczyć rozwiązanie i wyjaśnienie</summary>

**Poprawna odpowiedź: (B)** `sns.boxplot(data=pacjenci, x='oddzial', y='czas_pobytu_dni', hue='plec')`.

**Dlaczego B:**
- `boxplot` — bo pytanie jest o **rozkład** (czas pobytu), nie tylko średnią → barplot odrzucony
- `x='oddzial'` (kategoria), `y='czas_pobytu_dni'` (liczba) — typowe mapowanie
- `hue='plec'` — *aesthetics mapping* dający porównanie M/F **w obrębie każdego oddziału**

**Dlaczego nie A:** `plt.bar` pokaże tylko jeden kolor i nie podzieli na płeć. Dodatkowo barplot nie odpowiada na pytanie o rozkład.

**Dlaczego nie C:** heatmap korelacji pokaże tylko korelacje liczbowe (wiek vs czas_pobytu), pominie kategorie.

**Dlaczego nie D:** pairplot pokaże *wszystkie pary zmiennych* — szum informacyjny, gdy pytanie dotyczy jednej zależności.

**Sedno:** wybór wykresu pod *typ pytania*, nie pod *typ danych*. To jest praktyczne zastosowanie grammar of graphics i wiedzy o tym, że `pairplot` to figure-level (nie do siatki).

</details>

## 9. Dashboard — wszystko razem (kulminacja wykładu)

Wszystkie kawałki z poprzednich sekcji złączone w jedną figurę. Cel: **menedżer restauracji w 30 sekund wie wszystko o swoim biznesie**.

**Pytanie biznesowe:** *„Jak idzie biznes? Kiedy zarabiamy najwięcej? Kto zostawia największe napiwki? Czy są ukryte zależności?"*

> **Czytaj sekcjami `# === PANEL X ===` poniżej** — kod wygląda długo, ale to po prostu **6 niezależnych mini-przykładów** wklejonych do jednej `Figure` przez `GridSpec`. Każdy panel można uruchomić osobno (przekopiuj jego kod do nowej komórki) — to jest siła object-oriented API matplotlib.

In [ ]:
# Restaurant Analytics Dashboard — 6 paneli, jedna historia
fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#f8f9fa')

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# === PANEL 1: KPI główny — przychody per dzień ===
ax1 = fig.add_subplot(gs[0, :2])
tips_day = tips.groupby('day', observed=True).agg(
    total=('total_bill', 'sum'),
    count=('total_bill', 'count'),
).reset_index()
bars = ax1.bar(
    tips_day['day'].astype(str), tips_day['total'],
    color=sns.color_palette('muted')[:4],
    edgecolor='white', linewidth=1.2,
)
for bar, val in zip(bars, tips_day['total']):
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
        f'${val:.0f}', ha='center', va='bottom',
        fontsize=9, fontweight='bold',
    )
ax1.set_title('Łączne przychody wg dnia tygodnia', fontsize=12, fontweight='bold')
ax1.set_xlabel('Dzień')
ax1.set_ylabel('Suma rachunków (USD)')
ax1.set_facecolor('#ffffff')

# === PANEL 2: Udział wizyt — kołowy ===
ax2 = fig.add_subplot(gs[0, 2])
day_counts = tips['day'].value_counts()
ax2.pie(
    day_counts.values, labels=day_counts.index, autopct='%1.0f%%',
    colors=sns.color_palette('muted')[:len(day_counts)], startangle=90,
)
ax2.set_title('Udział wizyt wg dnia', fontsize=11, fontweight='bold')

# === PANEL 3: Boxplot napiwków ===
ax3 = fig.add_subplot(gs[1, 0])
sns.boxplot(data=tips, x='day', y='tip', hue='day', legend=False,
            palette='pastel', ax=ax3, linewidth=1.2)
ax3.set_title('Rozkład napiwków wg dnia', fontsize=11, fontweight='bold')
ax3.set_xlabel('Dzień')
ax3.set_ylabel('Napiwek (USD)')
ax3.set_facecolor('#ffffff')

# === PANEL 4: Heatmapa korelacji (z maska gornego trojkata — bo macierz symetryczna) ===
ax4 = fig.add_subplot(gs[1, 1])
corr = tips.select_dtypes('number').corr()  # lokalnie dla niezaleznosci komorki
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True   # ukryj gorny trojkat (z przekatna) — pokaz dolny
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax4, cbar=False, square=True,
)
ax4.set_title('Korelacja zmiennych (dolny trójkąt)', fontsize=11, fontweight='bold')

# === PANEL 5: Scatter rachunek vs napiwek ===
ax5 = fig.add_subplot(gs[1, 2])
sns.scatterplot(
    data=tips, x='total_bill', y='tip',
    hue='smoker', style='time', alpha=0.7, ax=ax5,
    palette={'Yes': '#e74c3c', 'No': '#2ecc71'},
)
ax5.set_title('Rachunek vs napiwek', fontsize=11, fontweight='bold')
ax5.set_xlabel('Rachunek (USD)')
ax5.set_ylabel('Napiwek (USD)')
ax5.set_facecolor('#ffffff')
ax5.legend(fontsize=8, title_fontsize=8)

# === PANEL 6: Violinplot rozkład rachunków (dolny — pełna szerokość) ===
ax6 = fig.add_subplot(gs[2, :])
sns.violinplot(
    data=tips, x='day', y='total_bill', hue='time', split=True,
    palette={'Lunch': '#3498db', 'Dinner': '#e74c3c'},
    ax=ax6, inner='box',
)
ax6.set_title('Rozkład rachunków wg dnia i pory dnia (Lunch vs Dinner)',
              fontsize=12, fontweight='bold')
ax6.set_xlabel('Dzień tygodnia')
ax6.set_ylabel('Rachunek (USD)')
ax6.set_facecolor('#ffffff')

# Tytuł nadrzędny
fig.suptitle('Restaurant Analytics Dashboard — Dataset Tips (244 rachunki)',
             fontsize=15, fontweight='bold', y=1.01)

plt.show()
plt.close()

**Anatomia dashboardu (od góry):**

| Panel | Pytanie biznesowe | Wykres | Dlaczego |
|-------|-------------------|--------|----------|
| 1 (główny) | *Kiedy zarabiamy najwięcej?* | Barplot z etykietami | KPI = jedna liczba per dzień |
| 2 (kołowy) | *Czy ruch jest równomierny?* | Pie | Proporcja całości |
| 3 (boxplot) | *Czy napiwki są stabilne?* | Boxplot | Mediana + outliery |
| 4 (heatmap) | *Co z czym idzie?* | Heatmap korelacji | Wszystkie pary zmiennych |
| 5 (scatter) | *Czy palacze inaczej napiwkują?* | Scatter z `hue` i `style` | 2 zmienne liczbowe + 2 kategorie |
| 6 (violin) | *Lunch czy kolacja generuje większy biznes?* | Violinplot split | Rozkład per pora dnia |

**Klucz dashboardu = NARRACJA, nie kolaż:**
- Panel 1 dominuje — to *główne KPI*, w które patrzy menedżer
- Panele 3-5 odpowiadają na *follow-up questions* (pytania uzupełniające)
- Panel 6 dolny zamyka pełną szerokością — *„najważniejsza obserwacja na finiszu"*

**Why this, why now, why not X:** dashboard z 6 paneli jest blisko granicy wytrzymałości poznawczej (George Miller 1956, *„The magical number seven, plus or minus two"* — magiczna liczba 7±2 elementów w pamięci roboczej). Więcej niż 8 paneli zaczyna szumieć. Mniej niż 4 daje za mało kontekstu. **6 to złoty środek dla raportów zarządczych.**

## 10. Eksport — savefig, adnotacje, profesjonalne wykończenie

Dashboard gotowy. Teraz zapisanie go do pliku, którego użyje menedżer.

In [ ]:
# Eksport do PNG (raster) i PDF (wektorowy) z parametrami publikacyjnymi
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=tips, x='day', y='total_bill', hue='sex',
    palette='muted', ax=ax, errorbar=('ci', 95),
)
ax.set_title('Średni rachunek wg dnia i płci', fontsize=13)
ax.set_xlabel('Dzień tygodnia')
ax.set_ylabel('Rachunek (USD)')
sns.despine()  # despine — usuwa górną i prawą krawędź osi (minimalistyczny styl)

plt.tight_layout()

# Eksport PNG (raster — bitmapowy, dobry do internetu)
plt.savefig(
    '/tmp/restauracja_demo.png',
    dpi=150,              # 72=web, 150=ekran HD, 300=druk
    bbox_inches='tight',  # bbox_inches='tight' — automatycznie dobiera marginesy
    facecolor='white',    # białe tło (zamiast przezroczystego)
    format='png',
)
print("PNG zapisany: /tmp/restauracja_demo.png")

# Eksport PDF (wektorowy — skaluje się bez utraty jakości)
plt.savefig(
    '/tmp/restauracja_demo.pdf',
    dpi=300,
    bbox_inches='tight',
    format='pdf',
)
print("PDF zapisany: /tmp/restauracja_demo.pdf")

plt.show()
plt.close()

**Trzy parametry savefig które zmieniają wszystko:**

1. **`dpi=150`** (*dots per inch* — punkty na cal, gęstość pikseli) — 72 do internetu, 150 do prezentacji HD, 300 do druku książkowego
2. **`bbox_inches='tight'`** — automatyczne marginesy. **Bez tego tytuł i etykiety bywają obcięte.** Prawie zawsze chcesz `'tight'`
3. **`facecolor='white'`** — białe tło zamiast domyślnego przezroczystego. Domyślne (przezroczyste) wygląda źle na szarych slajdach prezentacji

**Format pliku — kiedy który:**

| Format | Typ | Skalowanie | Użycie |
|--------|-----|------------|--------|
| **PNG** | raster (bitmapa) | obniża jakość | internet, slajdy, raporty |
| **PDF** | wektorowy | nieskończone | druk, LaTeX, książka |
| **SVG** | wektorowy | nieskończone | strony www, edycja w Inkscape/Illustrator |
| **JPG** | raster (kompresja stratna) | obniża jakość | NIE używaj do wykresów (artefakty kompresji)|

`sns.despine()` — usuwa górną i prawą ramkę osi (despine = *de-spine*, usuwanie kręgosłupa wykresu). Daje minimalistyczny, *Tufte-esque* (w stylu Edwarda Tufte) wygląd. Popularne w publikacjach naukowych.

### 10.1 Adnotacje — strzałka wskazująca ważny punkt

In [ ]:
# Adnotacja strzałką — wyróżnij szczyt dla menedżera
tips_sum = tips.groupby('day', observed=True)['total_bill'].sum().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    tips_sum['day'].astype(str), tips_sum['total_bill'],
    color=sns.color_palette('muted')[:4],
)

# Znajdź szczyt
max_idx = tips_sum['total_bill'].idxmax()
max_val = tips_sum.loc[max_idx, 'total_bill']

# Adnotacja ze strzałką (.annotate w matplotlib)
ax.annotate(
    f'Szczyt: ${max_val:.0f}',
    xy=(max_idx, max_val),                  # punkt strzałki
    xytext=(max_idx + 0.6, max_val + 150),  # pozycja tekstu
    arrowprops=dict(arrowstyle='->', color='red', lw=1.8),
    fontsize=11, color='red', fontweight='bold',
)

ax.set_title('Łączne przychody wg dnia — z adnotacją szczytu', fontsize=13)
ax.set_xlabel('Dzień tygodnia')
ax.set_ylabel('Suma rachunków (USD)')
sns.despine()

plt.tight_layout()
plt.show()
plt.close()

### 10.2 Legenda poza wykresem (gdy zasłania dane)

In [ ]:
# Legenda na zewnątrz wykresu — gdy kategorii dużo
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(
    data=tips, x='total_bill', y='tip',
    hue='day', style='time', ax=ax, alpha=0.8,
)

# bbox_to_anchor=(1.05, 1) — przesunięcie poza obszar wykresu
ax.legend(
    title='Dzień / Pora',
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0.,
)

ax.set_title('Scatter: rachunek vs napiwek (legenda poza wykresem)', fontsize=12)
ax.set_xlabel('Rachunek (USD)')
ax.set_ylabel('Napiwek (USD)')

plt.tight_layout()
plt.show()
plt.close()

### 10.3 Funkcja DRY — enkapsulacja wzorca

> **DRY** (Don't Repeat Yourself — nie powtarzaj się) — gdy używasz tego samego wzorca 3+ razy, wyodrębnij funkcję.

In [ ]:
def zbuduj_barplot(
    data, x, y, title, xlabel, ylabel,
    hue=None, palette='muted', filepath=None,
):
    """Buduje barplot seaborn z pełnymi etykietami i opcjonalnym eksportem."""
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=data, x=x, y=y, hue=hue, ax=ax, palette=palette,
                errorbar=('ci', 95))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    sns.despine()
    plt.tight_layout()
    if filepath:
        plt.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"Zapisano: {filepath}")
    plt.show()
    plt.close()


# Użycie — ten sam styl, różne dane, jedna linijka per wywołanie
zbuduj_barplot(
    tips, 'day', 'total_bill',
    'Średni rachunek wg dnia tygodnia',
    'Dzień', 'Rachunek (USD)',
    hue='sex',
)

zbuduj_barplot(
    tips, 'day', 'tip',
    'Średni napiwek wg dnia tygodnia',
    'Dzień', 'Napiwek (USD)',
    hue='smoker', palette='Set2',
)

## 11. Antywzorce wizualizacji — czego NIE robić

Tufte (*VDQI* 1983/2001) i Cairo (*The Truthful Art* 2016) zebrali listę grzechów wizualizacyjnych. Pięć najczęstszych w analityce biznesowej:

| Antywzorzec | Dlaczego źle | Co użyć zamiast |
|-------------|--------------|-----------------|
| **Pie chart >5 kategorii** | Mózg słabo szacuje kąty (Cleveland & McGill 1984) | Barplot — łatwo porównać długości |
| **Tęczowa paleta dla danych liczbowych** | Percepcyjnie nieliniowa (jet/rainbow) | viridis (sekwencyjna), coolwarm (dywergencyjna) |
| **Skala Y nie od 0 w barplot** | Wyolbrzymia różnice (manipulacja) | Skala Y zawsze od 0 dla bar; dla line OK od min |
| **3D bar/pie chart** | Perspektywa zniekształca proporcje | 2D z tymi samymi danymi |
| **Dual-axis chart (2 osie Y)** | Tworzy fałszywe korelacje wizualne | Subplot 2x1, dwa osobne wykresy |

> *„Pie charts are bad. The only thing worse than a pie chart is several of them."* — Edward Tufte
> *„Wykresy kołowe są złe. Jedyne co jest gorsze od wykresu kołowego to kilka wykresów kołowych."*

In [ ]:
# Demo antywzorca: skala Y nie od 0 w barplocie
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

day_means = tips.groupby('day', observed=True)['total_bill'].mean()

# LEWY: antywzorzec — skala Y zaczyna od 17 (wyolbrzymiona różnica)
axes[0].bar(day_means.index.astype(str), day_means.values, color='#e74c3c')
axes[0].set_ylim(17, 22)
axes[0].set_title('ANTYWZORZEC: oś Y od 17\n(różnice wyglądają na 4× większe niż są)',
                  fontsize=10, color='red')
axes[0].set_ylabel('Średni rachunek (USD)')

# PRAWY: poprawnie — od 0
axes[1].bar(day_means.index.astype(str), day_means.values, color='#2ecc71')
axes[1].set_ylim(0, 25)
axes[1].set_title('WZORZEC: oś Y od 0\n(różnice w prawdziwych proporcjach)',
                  fontsize=10, color='green')
axes[1].set_ylabel('Średni rachunek (USD)')

fig.suptitle('Manipulacja skalą Y — klasyczny grzech wizualizacji biznesowej',
             fontsize=12)
plt.tight_layout()
plt.show()
plt.close()

**Wniosek etyczny:** wybór wizualizacji to *nie tylko estetyka — to argument*. Niewłaściwy wykres może *kłamać* nie zmieniając ani jednej liczby. **Twoja praca jako analityka:** mówić prawdę przez dane, nie tylko pokazywać liczby.

## 12. Ściąga — wszystkie kluczowe komendy w jednym miejscu

```python
# === SETUP — raz na początku notebooka ===
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
sns.set_theme(style='whitegrid', palette='muted')

# === SEABORN — typy wykresów (axes-level: przyjmują ax=) ===
sns.barplot(data=df, x='kat', y='num', hue='grupa', ax=ax, errorbar=('ci', 95))
sns.boxplot(data=df, x='kat', y='num', hue='grupa', ax=ax)
sns.violinplot(data=df, x='kat', y='num', hue='kat2', split=True, ax=ax, inner='box')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
sns.scatterplot(data=df, x='n1', y='n2', hue='kat', style='kat2', alpha=0.7, ax=ax)
sns.countplot(data=df, x='kat', hue='kat2', ax=ax)
sns.stripplot(data=df, x='kat', y='num', jitter=True, alpha=0.5, ax=ax)

# === SEABORN — figure-level (NIE przyjmują ax=, własna figura) ===
g = sns.pairplot(df[cols], hue='kat', diag_kind='kde')      # macierz par
g = sns.jointplot(data=df, x='n1', y='n2', hue='kat')       # 1 para + marginalne
g = sns.catplot(data=df, x='kat', y='num', kind='box')      # facet po kolumnach

# === SUBPLOTS — regularna siatka ===
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes[0, 0]  # dostęp [wiersz, kolumna]

# === GRIDSPEC — nieregularna siatka ===
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)
ax = fig.add_subplot(gs[0, :])     # cały wiersz 0
ax = fig.add_subplot(gs[1, 0])     # wiersz 1, kolumna 0
ax = fig.add_subplot(gs[2, :2])    # wiersz 2, kolumny 0-1

# === SHARED AXES — wspólne osie ===
fig, axes = plt.subplots(2, 2, sharex='col', sharey='row')

# === KONWERSJA LONG <-> WIDE ===
long  = wide.melt(id_vars='id', var_name='zmienna', value_name='wartosc')
wide  = long.pivot_table(values='wartosc', index='id', columns='zmienna')

# === STYL I EKSPORT ===
fig.suptitle('Tytuł nadrzędny', fontsize=14, fontweight='bold')
sns.despine()                                                # minimalistyczny styl
plt.tight_layout()                                           # lub constrained_layout=True
plt.savefig('plik.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()  # ZAWSZE na końcu — zwalnia pamięć

# === ADNOTACJE ===
ax.annotate('Tekst', xy=(x, y), xytext=(x+1, y+1),
            arrowprops=dict(arrowstyle='->', color='red'))
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')        # legenda na zewnątrz
```

## 13. 🔬 Podgląd laboratorium L10

Za chwilę na laboratorium będziesz robić **dokładnie to samo, co przed chwilą widziałeś** — krok po kroku, w swoim tempie.

**Start labu (3 komendy w terminalu Windows):**
```cmd
cd C:\Users\TwojUzytkownik\python2_projekt
.venv\Scripts\activate
code .
```

**Co zrobicie na labie (90 min):**

| Czas | Ćwiczenie | Co konkretnie |
|------|-----------|---------------|
| 20 min | Ćw. 1 | Barplot, boxplot, violinplot, heatmap z datasetu tips |
| 20 min | Ćw. 2 | Subplots regularne 2×2 + GridSpec nieregularny + shared axes |
| 30 min | Ćw. 3 | **Pełny dashboard 4-6 paneli** — samodzielnie, własna narracja |
| 15 min | Ćw. 4 | Eksport `savefig()`, adnotacja, *commit* (zatwierdzenie zmian) do repo |

**Notebook labu:** `lab10_seaborn_dashboard.ipynb` (utworzysz sam)
**Dataset:** ten sam — `sns.load_dataset('tips')` (244 rachunki)
**Commit message** (komunikat zatwierdzenia w git): `L10: Dashboard Seaborn z datasetu tips — barplot, boxplot, heatmap, scatter`

**🔍 Pierwsze ćwiczenie 1a — dokładnie tak będzie wyglądać:**

```python
# Ćwiczenie 1a (lab) — barplot z hue
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset('tips')
sns.set_theme(style='whitegrid', palette='muted')

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=tips, x='day', y='total_bill', hue='sex',
    ax=ax, palette='muted', errorbar=('ci', 95),
)
ax.set_title('Średni rachunek wg dnia tygodnia i płci')
ax.set_xlabel('Dzień tygodnia')
ax.set_ylabel('Średni rachunek (USD)')
plt.tight_layout()
plt.show()
plt.close()

# Pytanie: w który dzień tygodnia mężczyźni płacą średnio najwięcej?
```

Tych 16 linii kodu napiszesz na labie samodzielnie — masz tu wzór ze ściągą.

**Zapowiedź W11 (kolejny tydzień):** *Statystyka opisowa z SciPy* — miary tendencji centralnej, rozproszenia, korelacja Pearsona/Spearmana, rozkład normalny. Dzisiejsza wizualizacja (heatmap korelacji, violinplot rozkładów) **wraca w nowej roli** — jako narzędzie diagnostyczne testów statystycznych.

## 14. Złota myśl na koniec

> *„The simple graph has brought more information to the data analyst's mind than any other device."*
> *„Prosty wykres wniósł więcej informacji do umysłu analityka niż jakiekolwiek inne narzędzie."*
>
> — **John W. Tukey** (1915-2000), twórca *Exploratory Data Analysis* (EDA — eksploracyjnej analizy danych), wynalazca *FFT* (Fast Fourier Transform — szybkiej transformaty Fouriera, fundamentu cyfrowego przetwarzania sygnałów), mentor pierwszych statystyków-programistów

**Myśl Feynmana o pierwszeństwie rzeczywistości nad PR-em** (*PR — Public Relations*, działania wizerunkowe; z raportu Komisji Rogersa o katastrofie wahadłowca *Challenger* w 1986, którego Feynman był członkiem):

> *„For a successful technology, reality must take precedence over public relations, for Nature cannot be fooled."*
> *„W udanej technologii rzeczywistość musi mieć pierwszeństwo nad PR-em, bo Natury oszukać się nie da."*

> *Pamiętasz cytat o oszukiwaniu samego siebie z W06? Ten jest jego rozwinięciem.* W kontekście wizualizacji: **manipulacja skalą Y, tęczowe palety, pie chart >5 kategorii** — możesz oszukać klienta na slajdzie, ale danych oszukać się nie da. Wcześniej czy później rzeczywistość wygra.

---

### 🎯 Pięć kluczowych zagadnień — powtórka

🔑 1. **Seaborn nie zastępuje matplotlib** — to wrapper. Pod spodem zawsze matplotlib.
🔑 2. **Dobierasz typ wykresu pod pytanie biznesowe** — nie pod „co ładnie wygląda".
🔑 3. **Long format → seaborn z `hue=`. Wide format → matplotlib `bar`. Konwersja: `melt()` / `pivot_table()`.**
🔑 4. **Dashboard = jedna figura, wiele wykresów, JEDNA HISTORIA.** GridSpec dla nieregularnych paneli.
🔑 5. **`savefig(..., dpi=150, bbox_inches='tight', facecolor='white')`** — bez tych 3 parametrów nie wygląda profesjonalnie.

---

**Zadanie domowe (nieoceniane):**
1. Znajdź **publiczny dataset CSV** (Kaggle, dane.gov.pl, GUS).
2. Wczytaj go pandasem.
3. **Sformułuj 1 pytanie biznesowe** (nie *„zrobię ładny wykres"*, tylko *„czy X zależy od Y w segmencie Z?"*).
4. Zbuduj **dashboard z 4 paneli** odpowiadający na to pytanie.
5. Eksportuj do PNG przez `savefig()`.
6. Commit do repo GitHub.

---

📅 **Następny wykład:** W11 — Statystyka opisowa z SciPy (miary, korelacja, rozkłady)
